# Why-What-How RAG: full run on Kaggle
Runs the three arms (Gemini alone, SLM hints without search, SLM hints with web search) on the same questions.

1. **Settings:** Accelerator **GPU T4 x1**, Internet **On**.
2. **Add-ons → Secrets:** add `GEMINI_API_KEY`, `SERPER_API_KEY`, `HF_TOKEN` (the HF account must have accepted the Llama-3.2 terms, and the GPQA terms if you run GPQA).
3. Run all. Each benchmark resumes where it stopped, so if Gemini's daily limit is hit, run again the next day.
4. Download `/kaggle/working/results.zip` at the end.

In [ ]:
import os, subprocess
if not os.path.exists('/kaggle/working/wwh'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Emminence23/why-what-how-rag', '/kaggle/working/wwh'], check=True)
os.chdir('/kaggle/working/wwh')
!pip install -q -r requirements.txt

In [ ]:
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
for k in ['GEMINI_API_KEY', 'SERPER_API_KEY', 'HF_TOKEN']:
    os.environ[k] = s.get_secret(k)

In [ ]:
# 3 benchmarks x 100 questions x 3 arms: ~900 Gemini calls and ~900 searches.
BENCHES = ['arc', 'hellaswag', 'musr']
MODE = 'instruction'
for b in BENCHES:
    !python run.py --bench {b} --mode {MODE} --arms base,nosearch,search --n 100

In [ ]:
!python analyze.py
import shutil
shutil.make_archive('/kaggle/working/results', 'zip', '.', 'results')